# Part 2 — Extending the series to 2024

Part 1 stops in 2000. This notebook carries every indicator forward to 2024 and
plots each against its GEAR target.

The series chains three different data vintages. We keep a source label on every
row and mark the join on every chart, rather than presenting one smooth line that
hides the break.

**Run Part 1 first** — this notebook reads the corrected file it produces.

## Setup

In [ ]:
from pathlib import Path
import pandas as pd

# Find the project folder: walk up from the working directory until we see a
# folder holding both resources/ and output/. This works whether the notebook is
# run from its own folder or from the repository root, so it keeps working after
# the project is moved or cloned onto another machine.
def find_project(start=None):
    here = Path(start or Path.cwd()).resolve()
    for folder in [here, *here.parents]:
        if (folder / "resources").is_dir() and (folder / "output").is_dir():
            return folder
    raise FileNotFoundError(
        "Could not find the project folder (the one containing 'resources' and "
        "'output'). Open this notebook from inside the project folder."
    )


PROJECT = find_project()

RES = PROJECT / "resources"                  # supplied data, never written to
BASE = PROJECT / "output"                    # everything we generate

print("Project folder:", PROJECT)

ADDITIONAL_DIR = RES / "data" / "Additional data"

YEARS = list(range(1996, 2025))
REQUIRED = [
    "real_gdp_growth_pct",
    "real_private_investment_growth_pct",
    "nonagric_formal_employment_growth_pct",
    "inflation_cpi_pct",
    "deficit_pct_gdp",
    "real_bank_rate_pct",
]

## 1996-2000, from Part 1

These are Nattrass's figures plus the 2000 values we sourced from SARB.

In [ ]:
# Load the corrected table Part 1 produced.
gear = pd.read_csv(BASE / "data" / "gear_target_vs_actual_1996_2000_CORRECTED.csv")
gear = gear[gear["indicator"].isin(REQUIRED)]

block_1996 = gear[["indicator", "year", "actual_to_1999", "gear_target"]].copy()
block_1996 = block_1996.rename(columns={"actual_to_1999": "actual"})
block_1996["source"] = "Nattrass (2001) Table 1 / SARB AER 2001"
block_1996["vintage_block"] = "1996-2000 (Nattrass / SARB AER2001)"

block_1996.head()

## 2001-2024, from SARB and StatsSA

Each series is read separately so its source stays attached to it.

In [ ]:
rows = []

# Real GDP growth, current-vintage SARB market prices.
gdp = pd.read_csv(ADDITIONAL_DIR / "sarb_gdp_marketprices_1999_2025_RAW.csv")
for _, r in gdp[gdp["year"].between(2001, 2024)].iterrows():
    rows.append(("real_gdp_growth_pct", int(r["year"]), r["sarb_growth_pct"],
                 "SARB, current-vintage real GDP at market prices (Additional data; "
                 "no series code given)"))

# Real private investment growth, private business enterprises only.
inv = pd.read_csv(ADDITIONAL_DIR / "sarb_private_investment_1994_2025_RAW.csv")
for _, r in inv[inv["year"].between(2001, 2024)].iterrows():
    rows.append(("real_private_investment_growth_pct", int(r["year"]), r["growth_pct"],
                 "SARB, real private business enterprise GFCF, current vintage "
                 "(Additional data; no series code given)"))

# Headline CPI, annual average.
cpi = pd.read_csv(ADDITIONAL_DIR / "statssa_cpi_1994_2025.csv")
for _, r in cpi[cpi["year"].between(2001, 2024)].iterrows():
    rows.append(("inflation_cpi_pct", int(r["year"]), r["cpi_annual_avg_pct"],
                 "StatsSA P0141, headline CPI, annual average (Additional data)"))

# Real bank rate: nominal Bank rate minus CPI, as Nattrass built hers.
rate = pd.read_csv(ADDITIONAL_DIR / "real_policy_rate_1994_2025.csv")
for _, r in rate[rate["year"].between(2001, 2024)].iterrows():
    rows.append(("real_bank_rate_pct", int(r["year"]), r["real_policy_rate_pct"],
                 "SARB Bank rate annual average minus StatsSA CPI (Additional data)"))

print(f"{len(rows)} observations so far")

### The deficit needs two adjustments

The file is fiscal years ending March, with deficits stored as negative. We flip
the sign so a deficit is positive, and map fiscal year ending March Y to calendar
year Y-1, because 9 of its 12 months fall in Y-1.

That mapping is a judgement call, not a fact, so we state it.

In [ ]:
# Deficit: flip sign so positive means in deficit, and shift to calendar years.
deficit = pd.read_csv(ADDITIONAL_DIR / "deficit_pct_gdp_1994_2025_FINAL.csv")
deficit["calendar_year"] = deficit["fiscal_year_ending_march"] - 1
deficit["deficit_positive"] = -deficit["deficit_pct_gdp"]

for _, r in deficit[deficit["calendar_year"].between(2001, 2024)].iterrows():
    rows.append(("deficit_pct_gdp", int(r["calendar_year"]), r["deficit_positive"],
                 f"SARB cash-flow balance / nominal GDP, FY ending March "
                 f"{int(r['fiscal_year_ending_march'])} mapped to calendar year "
                 f"{int(r['calendar_year'])} (Additional data; assumption stated in script)"))

deficit[["fiscal_year_ending_march", "calendar_year", "deficit_positive"]].head()

### Employment has a real gap

The labour data changes survey underneath us (OHS, then LFS, then QLFS, then QES).
These are not comparable, so we never splice them into one line.

The supplied extract also has nothing at all for 2016-2024. We leave that blank.
An interpolated line would look usable and be wrong.

In [ ]:
# This file has an unescaped comma inside one text field, which confuses pandas'
# column detection. Split each line manually, capping the number of splits.
with open(ADDITIONAL_DIR / "employment_series_SEGMENTED.csv", encoding="utf-8") as f:
    lines = [line.rstrip("\n") for line in f]

header = lines[0].split(",")
emp = pd.DataFrame([line.split(",", maxsplit=len(header) - 1)
                    for line in lines[1:] if line], columns=header)
emp["growth_pct"] = pd.to_numeric(emp["growth_pct"], errors="coerce")

# A survey dated March of year Y describes the year that just ended, so Y-1.
for _, r in emp.dropna(subset=["growth_pct"]).iterrows():
    date = str(r["date"])
    if "-03" in date:
        year = int(date.split("-")[0]) - 1
    elif "-12" in date:
        year = int(date.split("-")[0])
    else:
        continue
    if 2001 <= year <= 2024:
        rows.append(("nonagric_formal_employment_growth_pct", year, r["growth_pct"],
                     f"StatsSA {r['regime']}, {r['source']}"))

emp[["regime", "date", "growth_pct"]].head(10)

## Join the blocks together

In [ ]:
# Combine the two vintage blocks.
block_2001 = pd.DataFrame(rows, columns=["indicator", "year", "actual", "source"])
block_2001["gear_target"] = pd.NA
block_2001["vintage_block"] = "2001-2024 (Additional data / current vintage)"

combined = pd.concat([block_1996, block_2001], ignore_index=True, sort=False)

# Reindex onto every indicator-year so missing years show as blanks, not absences.
full = pd.MultiIndex.from_product([REQUIRED, YEARS], names=["indicator", "year"])
series = combined.set_index(["indicator", "year"]).reindex(full).reset_index()

(BASE / "data").mkdir(parents=True, exist_ok=True)
series.to_csv(BASE / "data" / "part2_extended_series_1996_2024.csv", index=False)

# How many years do we actually have for each indicator?
for ind in REQUIRED:
    have = series[series["indicator"] == ind].dropna(subset=["actual"])
    missing = sorted(set(YEARS) - set(have["year"]))
    print(f"{ind:42s} {len(have):2d}/29 years"
          + (f"  missing {missing[0]}-{missing[-1]}" if missing else ""))

## Plot each indicator against its target

Solid line is what happened, dashed is GEAR's target path, dotted is the 2000
target held flat as a steady-state benchmark. The grey line marks where the data
vintage changes.

In [ ]:
import matplotlib.pyplot as plt

TITLES = {
    "real_gdp_growth_pct": "Real GDP growth (%)",
    "real_private_investment_growth_pct": "Real private investment growth (%)",
    "nonagric_formal_employment_growth_pct": "Non-agric. formal employment growth (%)",
    "inflation_cpi_pct": "CPI inflation (%)",
    "deficit_pct_gdp": "Budget deficit (% of GDP)",
    "real_bank_rate_pct": "Real bank/repo rate (%)",
}

FIG_DIR = BASE / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

for indicator, title in TITLES.items():
    sub = series[series["indicator"] == indicator].sort_values("year")
    targets = sub.dropna(subset=["gear_target"])

    fig, ax = plt.subplots(figsize=(8, 4.5))
    # Gaps in the data break the line automatically; nothing is interpolated.
    ax.plot(sub["year"], sub["actual"], color="#1a5276", marker="o",
            markersize=3, linewidth=1.8, label="Actual")
    ax.plot(targets["year"], targets["gear_target"], color="#c0392b",
            linestyle="--", marker="s", markersize=3, linewidth=1.5,
            label="GEAR target (1996\u20132000)")

    final = targets["gear_target"].iloc[-1]
    ax.axhline(final, color="#c0392b", linestyle=":", linewidth=1, alpha=0.6,
               label=f"GEAR 2000 target ({final:.1f}) held flat")

    ax.axvline(2000.5, color="grey", linewidth=0.8, alpha=0.5)
    ax.text(2000.5, ax.get_ylim()[1], " vintage splice", fontsize=7,
            color="grey", va="top", rotation=90)

    ax.set_title(title)
    ax.set_xlabel("Year")
    ax.set_ylabel("%")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f"part2_{indicator}.png", dpi=150)
    plt.show()

## Which targets were actually met?

A target counts as met if the actual beat GEAR's target for that year, or after
2000, GEAR's final 2000 target treated as the steady-state goal. Higher is better
for growth, investment and employment; lower is better for inflation, the deficit
and the real interest rate.

In [ ]:
# True means a higher number is better.
HIGHER_IS_BETTER = {
    "real_gdp_growth_pct": True,
    "real_private_investment_growth_pct": True,
    "nonagric_formal_employment_growth_pct": True,
    "inflation_cpi_pct": False,
    "deficit_pct_gdp": False,
    "real_bank_rate_pct": False,
}

final_targets = series.dropna(subset=["gear_target"]).groupby("indicator")["gear_target"].last()

for indicator, higher in HIGHER_IS_BETTER.items():
    sub = series[series["indicator"] == indicator].dropna(subset=["actual"]).sort_values("year")
    met = []
    for _, r in sub.iterrows():
        target = r["gear_target"] if pd.notna(r["gear_target"]) else final_targets[indicator]
        if (r["actual"] >= target) if higher else (r["actual"] <= target):
            met.append(int(r["year"]))
    print(f"{TITLES[indicator]:42s} met {len(met):2d}/{len(sub)} years")
    print(f"{'':42s} {met if met else 'never'}")

## What this shows

The three stabilisation indicators (inflation, deficit, real interest rate) were
met in most years. The three growth indicators almost never were, and private
investment growth never once hit its target in 29 years.

GEAR's bet was that stability would buy investment. It bought stability.

One caveat worth keeping: growth did reach 5.6% in 2006 alongside a budget surplus
and the lowest inflation in the series, so this is not a simple story of failure.
Part 3 tests what was actually driving investment.